In [1]:
from openai import OpenAI
import os
from qdrant_client import QdrantClient
from qdrant_client.models import Filter,FieldCondition,MatchValue
from langsmith import Client
from dotenv import load_dotenv
load_dotenv()
import json

In [5]:
quadrant_client=QdrantClient(url="http://localhost:6333/")

In [3]:
clinet=OpenAI()

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


Download all data from Qadrant

In [16]:
all_points=quadrant_client.scroll(
    collection_name="Amazon_items_collection-00",
      limit=60,
    offset=None,
  
    with_payload=True,
    with_vectors=False
)

In [17]:
all_points[0][0].payload

{'description': 'JLab Go Air Sport - Wireless Workout Earbuds Featuring C3 Clear Calling, Secure Earhook Sport Design, 32+ Hour Bluetooth Playtime, and 3 EQ Sound Settings (Teal)LONG BATTERY LIFE: GO Air Sport features 8+ hours of playtime in each earbud and 24+ extra hours from the charging case (32+ hours total). Return to the case to recharge the buds, and use the integrated USB cable to charge the case.SWEATPROOF SPORT FIT: An IP55 sweat resistance holds up against sweat and dirt, so hit the trails, the gym and more. The ergonomic earhook fits even the smallest ears and stay securely and comfortably in place. Try all sizes of the gel tips to get a secure fit and just GO.DUAL CONNECT: Use either earbud independently or use both together. A built-in MEMS microphone in each bud ensures clear calls.EQ3 SOUND: With just a few taps via touch sensors, you can choose your sound preference: JLab Signature, Balanced and Bass Boost modes – without an app.TOUCH CONTROLS: Enjoy volume and track

In [18]:
all_context=[{"id":data.payload["parent_asin"],"text":data.payload["description"]} for data in all_points[0]]

render a prompty to generate synthetic Eval refernce dataset

In [19]:
output_schema={
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "question": {
        "type": "string",
        "description": "Suggested question"
      },
      "chunk_ids": {
        "type": "array",
        "items": {
          "type": "string",
          "description": "ID of the chunk that can be used to answer the question"
        }
      },
      "answer_example": {
        "type": "string",
        "description": "Suggested answer grounded in the context"
      },
      "reasoning": {
        "type": "string",
        "description": "Reasoning why the question can be answered using the specified chunks"
      }
    },
    
  }
}
SYSTEM_PROMPT=f"""
I am building a RAG application. I have a collection of 50 chunks of text.

The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.

I will provide all of the available products to you with IDs for each chunk.

I want you to come up with 30 questions to which the answers could be grounded in the chunk context.

The questions should imitate a potential real user of this RAG system.

As an output I need you to provide:
- The list of questions.
- The IDs of the chunks that could be used to answer them.
- An example answer to each question, grounded in the provided context.
- The reasoning why you chose those chunks to answer the question.

Constraints:
- 10 questions must require multiple chunks to answer.
- 15 questions must require only a single chunk to answer.
- 5 questions should intentionally **not** be answerable using the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

The output must be valid JSON and strictly follow the provided schema so that I can parse it programmatically.


"""

USER_PROMPT=f"""
here is the List of Chunks each list element is a dictionry with id and text

{all_context}

"""

In [20]:
print(USER_PROMPT)



here is the List of Chunks each list element is a dictionry with id and text

[{'id': 'B09VYGLLWH', 'text': 'JLab Go Air Sport - Wireless Workout Earbuds Featuring C3 Clear Calling, Secure Earhook Sport Design, 32+ Hour Bluetooth Playtime, and 3 EQ Sound Settings (Teal)LONG BATTERY LIFE: GO Air Sport features 8+ hours of playtime in each earbud and 24+ extra hours from the charging case (32+ hours total). Return to the case to recharge the buds, and use the integrated USB cable to charge the case.SWEATPROOF SPORT FIT: An IP55 sweat resistance holds up against sweat and dirt, so hit the trails, the gym and more. The ergonomic earhook fits even the smallest ears and stay securely and comfortably in place. Try all sizes of the gel tips to get a secure fit and just GO.DUAL CONNECT: Use either earbud independently or use both together. A built-in MEMS microphone in each bud ensures clear calls.EQ3 SOUND: With just a few taps via touch sensors, you can choose your sound preference: JLab Sig

In [ ]:
from openai import OpenAI

client = OpenAI()

res = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    temperature=0.2,
)

print(res.choices[0].message.content)

In [22]:
import json
json_output=res.choices[0].message.content

In [23]:
json_output=json.loads(json_output)

In [24]:
json_output

[{'question': 'What are the key features of the JLab Go Air Sport earbuds?',
  'chunk_ids': ['B09VYGLLWH'],
  'answer_example': 'The JLab Go Air Sport earbuds feature C3 Clear Calling, a secure earhook sport design, over 32 hours of Bluetooth playtime, 3 EQ sound settings, sweatproof IP55 rating, dual connect capability, touch controls, and include gel tips and a charging case.',
  'reasoning': "All key features are detailed in chunk B09VYGLLWH, which describes the earbuds' design, battery life, sound settings, and included accessories."},
 {'question': 'How long can the KT1 Ear Buds V5.3 Bluetooth earbuds play on a single charge?',
  'chunk_ids': ['B0BY7TRDXD'],
  'answer_example': 'The KT1 Ear Buds V5.3 can be used for 7 hours after a full charge, with a total of 120 hours of playtime using the charging case.',
  'reasoning': "Chunk B0BY7TRDXD provides specific details about the earbuds' playtime and the capacity of the charging case."},
 {'question': 'What are the specifications of 

In [25]:
len(json_output)

27

In [34]:

def get_description(parent_asin: str) -> str:
    
    points, next_page_offset = quadrant_client.scroll(
        collection_name="Amazon_items_collection-00",  
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=1,  
        with_payload=True,
        with_vectors=False
    )
    
   
    if points:
        payload = points[0].payload
       
        return payload.get("description", "Description key not found in payload.")
        
    return "No matching record found for this parent_asin."

In [35]:
res=get_description("B0BRR2R8HH")

In [36]:
res

'Gigabyte GeForce RTX 4070 Ti Gaming OC 12G Graphics Card, 3X WINDFORCE Fans, 12GB 192-bit GDDR6X, GV-N407TGAMING OC-12GD Video CardPowered by NVIDIA DLSS 3.Digital max resolution: 7680x4320NVIDIA Ada Lovelace Streaming Multiprocessors: Up to 2x performance and power efficiency4th Generation Tensor Cores: Up to 2x AI performance3rd Generation RT Cores: Up to 2x ray tracing performancePowered by GeForce RTX 4070 Ti 12GBIntegrated with 12GB GDDR6X 192-bit memory interfaceWINDFORCE Cooling System, RGB Fusion, Dual BIOS, Protection metal back plate, Anti-Sag Bracket'

### Create Eval Dataset in Langchain

In [40]:
lang_client=Client(api_key=os.environ['LANGSMITH_API_KEY'])

In [42]:
data_set_name="rag_eval2"
datset=lang_client.create_dataset(
    dataset_name=data_set_name,
    description="Dataset for Evaluation Rag pipeline"
)

In [43]:
datset.id

UUID('7f2ed3e7-ff32-4eb7-810c-0fde541fc996')

In [47]:
for item in json_output:
    lang_client.create_example(
        dataset_id=datset.id,
        inputs={"question":item['question']},
        outputs={
            "ground_truth": item['answer_example'],
            "refrence_context_ids":item["chunk_ids"],
            "reference_description":[get_description(id) for id in item["chunk_ids"]]


        }

    )